# Chapter 14: Advanced File Processing and Data Handling

Companion notebook to **python-from-zero** · `lesson-14` · based on *Python for VLSI*, Chapter 14.

### You will learn

- Parse line-structured reports with `split`/`strip`.
- Handle CSV with `split(',')` and the `csv` module.
- Extract values robustly with regular expressions.
- Build a small command-line interface using `argparse`.

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## Sample STA report

We regenerate the input file each run so the notebook is reproducible.

In [2]:
from pathlib import Path
Path("sta.rpt").write_text('''Startpoint: U1/clk
Endpoint:   U2/data
Path Group: clk
slack -0.045 setup violation
slack  0.120 hold    met
slack -0.089 setup   violation
slack  0.031 hold    met
slack -0.002 setup   violation
''')
print(Path("sta.rpt").read_text())


Startpoint: U1/clk
Endpoint:   U2/data
Path Group: clk
slack -0.045 setup violation
slack  0.120 hold    met
slack -0.089 setup   violation
slack  0.031 hold    met
slack -0.002 setup   violation



## Splitting lines into tokens

`.split()` on whitespace is the bread-and-butter of report parsing.

In [3]:
with open("sta.rpt") as f:
    for line in f:
        tok = line.strip().split()
        if tok and tok[0] == "slack":
            print(f"{tok[2]:5s} slack = {float(tok[1]):+.3f} ns")


setup slack = -0.045 ns
hold  slack = +0.120 ns
setup slack = -0.089 ns
hold  slack = +0.031 ns
setup slack = -0.002 ns


## Filtering lines

`startswith`, `endswith`, and `in` cover most filters.

In [4]:
with open("sta.rpt") as f:
    for line in f:
        if line.startswith("slack") and "violation" in line:
            print("violation:", line.strip())


violation: slack -0.045 setup violation
violation: slack -0.089 setup   violation
violation: slack -0.002 setup   violation


## CSV with the csv module

For quoted or escaped CSVs, `csv.reader` is safer than `split(',')`.

In [5]:
from pathlib import Path
import csv

Path("nets.csv").write_text('''clk,0.2,high_fanout
reset,0.1,low_fanout
data,0.25,high_fanout
''')

with open("nets.csv", newline="") as f:
    for row in csv.reader(f):
        net, fo, kind = row
        print(f"{net:6s} fanout={float(fo):.2f} ({kind})")


clk    fanout=0.20 (high_fanout)
reset  fanout=0.10 (low_fanout)
data   fanout=0.25 (high_fanout)


## Regex extraction

When lines vary, anchor to a compiled pattern instead of token index.

In [6]:
import re
from pathlib import Path

Path("sta_regex.rpt").write_text('''slack: -0.045 (VIOLATION)
slack:  0.110 (OK)
slack: -0.002 (VIOLATION)
''')

pat = re.compile(r"slack:\s*(-?\d+\.\d+)\s*\((\w+)\)")
with open("sta_regex.rpt") as f:
    for line in f:
        m = pat.search(line)
        if m:
            print(f"slack={float(m.group(1)):+.3f}  state={m.group(2)}")


slack=-0.045  state=VIOLATION
slack=+0.110  state=OK
slack=-0.002  state=VIOLATION


## Guarding bad input

Wrap risky conversions in `try/except` so one malformed line doesn't kill the whole run.

In [7]:
raw = ["slack 0.10 setup met",
       "slack abc setup violation",
       "",
       "slack -0.08 setup violation"]
good = []
for line in raw:
    tok = line.split()
    if not tok or tok[0] != "slack":
        continue
    try:
        good.append(float(tok[1]))
    except (ValueError, IndexError) as e:
        print("skipping:", line, "->", e)
print("clean:", good)


skipping: slack abc setup violation -> could not convert string to float: 'abc'
clean: [0.1, -0.08]


## argparse basics

`argparse` turns a script into a real CLI. We drive it from a fake argv list so the notebook stays self-contained.

In [8]:
import argparse
p = argparse.ArgumentParser(description="Summarize an STA report")
p.add_argument("report")
p.add_argument("--top", type=int, default=3)
p.add_argument("--verbose", action="store_true")
args = p.parse_args(["sta.rpt", "--top", "2", "--verbose"])
print("parsed:", vars(args))


parsed: {'report': 'sta.rpt', 'top': 2, 'verbose': True}


## Running a real CLI script

Wrap the parser in `main(argv=None)` and guard with `if __name__ == '__main__':`.

In [9]:
from pathlib import Path
import runpy, sys

Path("sta_cli.py").write_text('''import argparse

def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("report")
    p.add_argument("--top", type=int, default=3)
    args = p.parse_args(argv)

    slacks = []
    with open(args.report) as f:
        for line in f:
            tok = line.split()
            if tok and tok[0] == "slack":
                try: slacks.append(float(tok[1]))
                except ValueError: pass
    slacks.sort()
    print("worst", args.top, ":", slacks[:args.top])

if __name__ == "__main__":
    main()
''')

saved = sys.argv
try:
    sys.argv = ["sta_cli.py", "sta.rpt", "--top", "3"]
    runpy.run_path("sta_cli.py", run_name="__main__")
finally:
    sys.argv = saved


worst 3 : [-0.089, -0.045, -0.002]


## VLSI practice — end-to-end report triage

Combine parsing + regex + argparse into a tiny triage tool that prints text or JSON.

In [10]:
import argparse, json, re
from pathlib import Path

def triage(report_path, top=3):
    pat = re.compile(r"^slack\s+(-?\d+\.\d+)\s+(\w+)")
    records = []
    for line in Path(report_path).read_text().splitlines():
        m = pat.match(line.strip())
        if m:
            records.append({"slack": float(m.group(1)), "kind": m.group(2)})
    viols = sorted((r for r in records if r["slack"] < 0),
                   key=lambda r: r["slack"])
    return {"total_paths": len(records), "violations": len(viols), "worst": viols[:top]}

def main(argv=None):
    p = argparse.ArgumentParser(description="Triage an STA report")
    p.add_argument("report")
    p.add_argument("--top", type=int, default=3)
    p.add_argument("--json", dest="as_json", action="store_true")
    args = p.parse_args(argv)
    r = triage(args.report, top=args.top)
    if args.as_json:
        print(json.dumps(r, indent=2))
    else:
        print(f"paths      : {r['total_paths']}")
        print(f"violations : {r['violations']}")
        for i, x in enumerate(r["worst"], 1):
            print(f"  {i}. {x['kind']:5s} slack={x['slack']:+.3f}")

main(["sta.rpt", "--top", "3"])
print("--- JSON form ---")
main(["sta.rpt", "--top", "3", "--json"])


paths      : 5
violations : 3
  1. setup slack=-0.089
  2. setup slack=-0.045
  3. setup slack=-0.002
--- JSON form ---
{
  "total_paths": 5,
  "violations": 3,
  "worst": [
    {
      "slack": -0.089,
      "kind": "setup"
    },
    {
      "slack": -0.045,
      "kind": "setup"
    },
    {
      "slack": -0.002,
      "kind": "setup"
    }
  ]
}


### Exercise 1
Print lines of `sta.rpt` that contain both `setup` and `violation`.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
with open('sta.rpt') as f:
    for l in f:
        if 'setup' in l and 'violation' in l:
            print(l.strip())
```

</details>

### Exercise 2
Parse `nets.csv` into a dict `{net: fanout}`.

In [12]:
# Your code here


<details><summary>Show solution</summary>

```python
import csv
with open('nets.csv') as f:
    d = {r[0]: float(r[1]) for r in csv.reader(f)}
print(d)
```

</details>

### Exercise 3
Write an argparse parser with `--budget FLOAT` default `-0.01` and test it.

In [13]:
# Your code here


<details><summary>Show solution</summary>

```python
import argparse
p=argparse.ArgumentParser(); p.add_argument('--budget', type=float, default=-0.01)
print(p.parse_args(['--budget','-0.02']))
```

</details>

## Recap

- `.split()`/`.strip()` handle most line-structured reports.
- Reach for `csv` for anything beyond trivial comma files.
- Compile regex patterns and anchor them against variable lines.
- `argparse` is the stdlib way to build real CLIs.

## What's next

You've reached the end of the companion series. Return to the [python-from-zero](https://python-from-zero.pages.dev) site to keep going.